In [7]:
import torch
import torch.nn as nn
from torchinfo import summary

## EfficientNet

In [8]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction):
        super().__init__()

        hidden = channels // reduction

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.layer = nn.Sequential(
            nn.Conv2d(channels, hidden, 1),
            nn.SiLU(),

            nn.Conv2d(hidden, channels, 1),
            nn.Sigmoid()
        )
        
    def forward(self, X):
        scale = self.pool(X)
        scale = self.layer(scale)

        return X * scale
        

In [9]:
class MBConv(nn.Module):
    def __init__(self, in_channels, out_channels, expand_ratio, stride=1):
        super().__init__()

        hidden_layer = in_channels * expand_ratio

        self.expand = nn.Sequential(
            nn.Conv2d(in_channels, hidden_layer, 1, bias=False),
            nn.BatchNorm2d(hidden_layer),
            nn.SiLU()
        )

        self.depthwise = nn.Sequential(
            nn.Conv2d(
                hidden_layer,
                hidden_layer,
                3,
                stride=stride,
                padding=1,
                bias=False,
                groups=hidden_layer
            ),
            nn.BatchNorm2d(hidden_layer),
            nn.SiLU()
        )

        self.se = SEBlock(hidden_layer, 4)

        self.project = nn.Sequential(
            nn.Conv2d(hidden_layer, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels)
        )

        self.use_residual = (
            stride == 1 and in_channels == out_channels
        )

    def forward(self, x):

        identity = x

        x = self.expand(x)
        x = self.depthwise(x)
        x = self.se(x)
        x = self.project(x)

        if self.use_residual:
            x = x + identity

        return x

In [11]:
class EfficientNetB0(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(
                3,
                32,
                kernel_size=3,
                stride=2,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(32),
            nn.SiLU()
        )

        self.features = nn.Sequential(

            # Stage 1
            MBConv(
                32, 16,
                expand_ratio=1,
                stride=1
            ),

            # Stage 2
            MBConv(
                16, 24,
                expand_ratio=6,
                stride=2
            ),

            MBConv(
                24, 24,
                expand_ratio=6,
                stride=1
            ),

            # Stage 3
            MBConv(
                24, 40,
                expand_ratio=6,
                stride=2
            ),

            MBConv(
                40, 40,
                expand_ratio=6,
                stride=1
            ),

            # Stage 4
            MBConv(
                40, 80,
                expand_ratio=6,
                stride=2
            ),

            MBConv(
                80, 80,
                expand_ratio=6,
                stride=1
            ),

            MBConv(
                80, 80,
                expand_ratio=6,
                stride=1
            ),

            # Stage 5
            MBConv(
                80, 112,
                expand_ratio=6,
                stride=1
            ),

            MBConv(
                112, 112,
                expand_ratio=6,
                stride=1
            ),

            MBConv(
                112, 112,
                expand_ratio=6,
                stride=1
            ),

            # Stage 6
            MBConv(
                112, 192,
                expand_ratio=6,
                stride=2
            ),

            MBConv(
                192, 192,
                expand_ratio=6,
                stride=1
            ),

            MBConv(
                192, 192,
                expand_ratio=6,
                stride=1
            ),

            MBConv(
                192, 192,
                expand_ratio=6,
                stride=1
            ),

            # Stage 7
            MBConv(
                192, 320,
                expand_ratio=6,
                stride=1
            )
        )

        self.head = nn.Sequential(
            nn.Conv2d(
                320,
                1280,
                kernel_size=1,
                bias=False
            ),
            nn.BatchNorm2d(1280),
            nn.SiLU()
        )

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Linear(
            1280,
            num_classes
        )

    def forward(self, x):

        x = self.stem(x)
        x = self.features(x)
        x = self.head(x)

        x = self.pool(x)

        x = torch.flatten(x, 1)

        x = self.classifier(x)

        return x



model = EfficientNetB0().to("cuda")
summary(model, input_size=(1, 3, 224, 224))


Layer (type:depth-idx)                        Output Shape              Param #
EfficientNetB0                                [1, 10]                   --
├─Sequential: 1-1                             [1, 32, 112, 112]         --
│    └─Conv2d: 2-1                            [1, 32, 112, 112]         864
│    └─BatchNorm2d: 2-2                       [1, 32, 112, 112]         64
│    └─SiLU: 2-3                              [1, 32, 112, 112]         --
├─Sequential: 1-2                             [1, 320, 7, 7]            --
│    └─MBConv: 2-4                            [1, 16, 112, 112]         --
│    │    └─Sequential: 3-1                   [1, 32, 112, 112]         1,088
│    │    └─Sequential: 3-2                   [1, 32, 112, 112]         352
│    │    └─SEBlock: 3-3                      [1, 32, 112, 112]         552
│    │    └─Sequential: 3-4                   [1, 16, 112, 112]         544
│    └─MBConv: 2-5                            [1, 24, 56, 56]           --
│    │    └─S